# snRNA-seq Preprocessing

## Description
This pipeline preprocesses single-nuclei RNA-seq data in two stages. First it performs quality control with SCTK and Seurat: removing low-quality cells by mitochondrial content, total counts (nUMI) and detected genes (nFeature), filtering ambient RNA contamination with decontX, and removing doublets with a user-selected method (scds, scrublet, doubletFinder, or doubletCells). It then annotates cell types on the QC-filtered data using SingleR against a reference, transferring cluster-majority labels to the full Seurat object so the result is ready for pseudobulk aggregation.

## Input Files

| File | Description |
|------|-------------|
| `input/snrnaseq/protocol_example.snrnaseq.cellranger/` | CellRanger output directory (contains the per-sample counts folder with `filtered_feature_bc_matrix`) |
| `input/snrnaseq/protocol_example.snrnaseq.id_mapping.csv` | Sample barcode-to-patient ID mapping (columns: `libraryBatch`, `cellBarcode`, `individualID`) |
| `input/snrnaseq/protocol_example.snrnaseq.seurat_ref_SE.rds` | SingleR reference SummarizedExperiment object (used in Step 2) |

## Steps

### Step 1. QC filtering (sctk_qc)
Runs SCTK/Seurat QC on the CellRanger counts: it drops cells failing the mitochondrial-percent, minimum-count (nUMI) and minimum-gene (nFeature) thresholds, removes ambient RNA with decontX, and removes doublets with the chosen method (default `scds`). The sample-meta CSV maps cell barcodes to individual IDs. The result is a QC-filtered, clustered Seurat object.

In [ ]:
sos run pipeline/snRNAseq_preprocessing.ipynb sctk_qc \
    --input-dir input/snrnaseq/protocol_example.snrnaseq.cellranger \
    --output-dir output/snrna_seq \
    --sample-meta input/snrnaseq/protocol_example.snrnaseq.id_mapping.csv

### Step 2. Cell type annotation (cell_annotation)
Annotates cell types on the QC-filtered object using SingleR against the reference SummarizedExperiment. Cells are downsampled per cluster (default 1000) for the SingleR call, then a cluster-majority label is assigned when at least `cluster_threshold` (default 0.90) of the cluster agrees, and transferred back to the full Seurat object.

In [ ]:
sos run pipeline/snRNAseq_preprocessing.ipynb cell_annotation \
    --sctk-rds output/snrna_seq/SCTK_results/filtered_seuratobj.rds \
    --output-dir output/snrna_seq \
    --seurat-ref input/snrnaseq/protocol_example.snrnaseq.seurat_ref_SE.rds

## Output Files

| File | Description |
|------|-------------|
| `output/snrna_seq/SCTK_results/filtered_seuratobj.rds` | QC-filtered, clustered Seurat object (from Step 1) |
| `output/snrna_seq/QC_table/SCTK_QC_table.csv` | Per-sample SCTK QC metrics |
| `output/snrna_seq/SCTK_results/QC_summary.csv` | Cell/gene counts at each QC step |
| `output/snrna_seq/annotated_seuratobj.rds` | Downsampled Seurat object with per-cell SingleR labels (from Step 2) |
| `output/snrna_seq/celltyped_seuratobj.rds` | Full Seurat object with cluster-majority cell type labels (from Step 2) |

## Anticipated Results
Step 1 yields a QC-filtered Seurat object plus QC tables summarizing how many cells and genes survive each filtering step. Step 2 adds cell type labels, producing a celltype-labeled Seurat object ready for downstream pseudobulk aggregation.